In [1]:
from pathlib import Path
import pandas as pd

# Set the final master table as the only source for feature engineering.
# This keeps all new features traceable to the validated Group 7 dataset.

base_dir = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed"
)

master_path = (
    base_dir
    / "07_ward_master_dataset"
    / "ward_master_dataset_2011_2021.csv"
)

output_dir = base_dir / "08_feature_engineered_dataset"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    output_dir
    / "ward_features_2011_2021.csv"
)

print("Master file exists:", master_path.exists())
print("Output folder:", output_dir)
print("Output path:", output_path)

Master file exists: True
Output folder: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\08_feature_engineered_dataset
Output path: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\08_feature_engineered_dataset\ward_features_2011_2021.csv


In [2]:
# Load the validated Group 7 master table as the only input for feature engineering.
# We inspect the election and context fields before creating any new model features.

ward_features = pd.read_csv(
    master_path,
    encoding="utf-8-sig"
)

print("Master shape:", ward_features.shape)

print("\nElection fields:")
print([
    column for column in ward_features.columns
    if column in [
        "Province",
        "Municipality",
        "MunicipalityCode",
        "CurrentMunicipality",
        "Ward",
        "ElectionYear",
        "TurnoutRate",
        "RegisteredVoters",
        "BoundaryConsistent"
    ]
])

print("\nPoverty-related columns:")
print([
    column for column in ward_features.columns
    if any(
        term in column.lower()
        for term in ["poverty", "lbpl", "ubpl", "food poverty"]
    )
])

Master shape: (2599, 47)

Election fields:
['Province', 'Municipality', 'Ward', 'RegisteredVoters', 'ElectionYear', 'TurnoutRate', 'BoundaryConsistent', 'MunicipalityCode', 'CurrentMunicipality']

Poverty-related columns:
['Food poverty headcount (FPL, %)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)']


There are no literal LBPL or UBPL columns in Group 7. The available poverty fields are FPL/P0/P1/P2/share measures. So we should not fabricate an LBPL/UBPL pairing feature. Instead, we'll make the check explicit: verify whether those fields exist, record that they are unavailable, and leave the existing poverty context unchanged.

Before engineering the temporal features, we need to establish the ward identity key and inspect boundary consistency by election year. This is the critical safeguard for previous-turnout calculations.

In [3]:
# Build the historical ward key using municipality code and ward so wards are compared within their municipality.
# We check boundary consistency before carrying historical turnout forward between election periods.

ward_features["WardKey"] = (
    ward_features["MunicipalityCode"].astype(str).str.strip()
    + "_"
    + ward_features["Ward"].astype(str).str.strip()
)

print("Unique ward keys:", ward_features["WardKey"].nunique())

print("\nElection years:")
print(sorted(ward_features["ElectionYear"].unique()))

print("\nBoundary consistency by election year:")
print(
    ward_features
    .groupby("ElectionYear")["BoundaryConsistent"]
    .value_counts(dropna=False)
)

print("\nBoundary-inconsistent rows:",
      (~ward_features["BoundaryConsistent"]).sum())

Unique ward keys: 1011

Election years:
[np.int64(2011), np.int64(2016), np.int64(2021)]

Boundary consistency by election year:
ElectionYear  BoundaryConsistent
2011          True                  717
              False                 111
2016          True                  717
              False                 153
2021          True                  717
              False                 184
Name: count, dtype: int64

Boundary-inconsistent rows: 448


Good. This is an important result and it changes how we handle the historical features.

We have 1,011 unique municipality-code + ward keys, but BoundaryConsistent is False for 448 ward-year records. Therefore, we should not blindly use shift() across the entire dataset.

Also notice that 717 wards are boundary-consistent in every election year. Those are the safest records for temporal comparison.

For the feature engineering, we'll use a strict rule:

A previous-period feature is calculated only when the current and previous records share the same WardKey and both records are BoundaryConsistent == True

In [5]:
# Sort each ward's election history so consecutive elections can be compared in the correct order.
# Previous-period features will only use records that pass the boundary-consistency check.

ward_features = ward_features.sort_values(
    ["WardKey", "ElectionYear"]
).reset_index(drop=True)

ward_features["PreviousElectionYear"] = (
    ward_features.groupby("WardKey")["ElectionYear"]
    .shift(1)
)

print("Election year sequence counts:")
print(
    ward_features[
        ["ElectionYear", "PreviousElectionYear"]
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRows with a previous election record:",
      ward_features["PreviousElectionYear"].notna().sum())

print("Rows without a previous election record:",
      ward_features["PreviousElectionYear"].isna().sum())

Election year sequence counts:
ElectionYear  PreviousElectionYear
2011          NaN                     828
2016          2011.0                  717
              NaN                     153
2021          2011.0                    1
              2016.0                  870
              NaN                      30
Name: count, dtype: int64

Rows with a previous election record: 1588
Rows without a previous election record: 1011


Excellent — and this output reveals an important issue we need to fix before creating the previous-turnout feature.

For example, one 2021 record currently has PreviousElectionYear = 2011, because that ward key has no 2016 record. We must not treat 2011 as the previous election for 2021. The solution specifically means the immediately preceding election period:

2011: no previous turnout
2016: previous = 2011
2021: previous = 2016
If a 2021 ward has no 2016 record → previous turnout stays blank.

So rather than relying on shift() alone, we'll enforce the expected election-year relationship

In [6]:
# Keep only genuine consecutive election periods for historical feature calculations.
# This prevents a missing 2016 ward from incorrectly linking its 2021 turnout back to 2011.

expected_previous_year = {
    2011: None,
    2016: 2011,
    2021: 2016
}

ward_features["ExpectedPreviousYear"] = (
    ward_features["ElectionYear"].map(expected_previous_year)
)

ward_features["ConsecutiveElection"] = (
    ward_features["PreviousElectionYear"]
    == ward_features["ExpectedPreviousYear"]
)

print("Consecutive previous-election matches:")
print(
    ward_features[
        "ConsecutiveElection"
    ]
    .value_counts(dropna=False)
)

print("\nNon-consecutive previous records excluded:",
      (
          ward_features["PreviousElectionYear"].notna()
          & ~ward_features["ConsecutiveElection"]
      ).sum())

Consecutive previous-election matches:
ConsecutiveElection
True     1587
False    1012
Name: count, dtype: int64

Non-consecutive previous records excluded: 1


Perfect. Cell 05 confirms only 1 record had a non-consecutive historical match, and we're excluding it.

Now we need to add the boundary-consistency condition. A consecutive year alone isn't enough: both the current record and its previous record must be boundary-consistent.

We'll first create a safe flag identifying rows where the previous record is both:

the correct consecutive election period, and
boundary-consistent.

In [8]:
#***************************************************************************
#***************************************************************************

# Mark historical comparisons as safe only when the election periods are consecutive and the current ward is boundary-consistent.
# This prevents previous turnout and registration features from crossing unreliable ward boundaries.

ward_features["SafeHistoricalComparison"] = (
    ward_features["ConsecutiveElection"]
    & ward_features["BoundaryConsistent"]
)

print("Safe historical comparison rows:")
print(
    ward_features["SafeHistoricalComparison"]
    .value_counts(dropna=False)
)

print("\nSafe comparisons by election year:")
print(
    ward_features[
        ward_features["SafeHistoricalComparison"]
    ]
    .groupby("ElectionYear")
    .size()
)

print("\nRows excluded from historical comparison:",
      (~ward_features["SafeHistoricalComparison"]).sum())

Safe historical comparison rows:
SafeHistoricalComparison
True     1434
False    1165
Name: count, dtype: int64

Safe comparisons by election year:
ElectionYear
2016    717
2021    717
dtype: int64

Rows excluded from historical comparison: 1165


The important result is:

717 safe 2011 → 2016 comparisons.
717 safe 2016 → 2021 comparisons.
1,165 rows are excluded from historical comparison.
2011 has no possible previous election, as expected.
The 717 + 717 pattern matches the wards that were boundary-consistent across consecutive periods.

Now we can create the actual Previous Turnout feature.

We need to be careful here: shift() can identify the previous row, but we already established that only consecutive and boundary-consistent records are safe. We'll therefore only populate PreviousTurnout for SafeHistoricalComparison == True.

In [9]:
# Carry forward turnout only from the immediately preceding election for a safe ward comparison.
# All other rows stay blank because there is no reliable previous turnout to use.

ward_features["PreviousTurnout"] = pd.NA

safe_rows = ward_features["SafeHistoricalComparison"]

ward_features.loc[safe_rows, "PreviousTurnout"] = (
    ward_features.groupby("WardKey")["TurnoutRate"]
    .shift(1)
    .loc[safe_rows]
)

print("Previous turnout populated:",
      ward_features["PreviousTurnout"].notna().sum())

print("Previous turnout blank:",
      ward_features["PreviousTurnout"].isna().sum())

print("\nPrevious turnout by election year:")
print(
    ward_features
    .groupby("ElectionYear")["PreviousTurnout"]
    .apply(lambda x: x.notna().sum())
)

print("\n2011 previous turnout populated:",
      ward_features.loc[
          ward_features["ElectionYear"] == 2011,
          "PreviousTurnout"
      ].notna().sum()
)

Previous turnout populated: 1434
Previous turnout blank: 1165

Previous turnout by election year:
ElectionYear
2011      0
2016    717
2021    717
Name: PreviousTurnout, dtype: int64

2011 previous turnout populated: 0


In [10]:
# We have 717 reliable previous-turnout comparisons for 2016 and 717 for 2021, while 2011 correctly remains blank
# Calculate how many registered voters each ward gained or lost since the previous election.
# This helps capture changes in the ward's registered-voter base over time.

ward_features["RegisteredVotersChange"] = pd.NA

previous_registered = (
    ward_features.groupby("WardKey")["RegisteredVoters"]
    .shift(1)
)

ward_features.loc[safe_rows, "RegisteredVotersChange"] = (
    ward_features.loc[safe_rows, "RegisteredVoters"]
    - previous_registered.loc[safe_rows]
)

print("Registered voter change populated:",
      ward_features["RegisteredVotersChange"].notna().sum())

print("Registered voter change blank:",
      ward_features["RegisteredVotersChange"].isna().sum())

print("\nRegistered voter change by election year:")
print(
    ward_features
    .groupby("ElectionYear")["RegisteredVotersChange"]
    .apply(lambda x: x.notna().sum())
)

print("\n2011 registered voter change populated:",
      ward_features.loc[
          ward_features["ElectionYear"] == 2011,
          "RegisteredVotersChange"
      ].notna().sum())

Registered voter change populated: 1434
Registered voter change blank: 1165

Registered voter change by election year:
ElectionYear
2011      0
2016    717
2021    717
Name: RegisteredVotersChange, dtype: int64

2011 registered voter change populated: 0


The numbers line up with the previous-turnout feature:

1,434 safe ward comparisons populated
1,165 left blank
2016: 717 comparisons
2021: 717 comparisons
2011: 0, correctly blank

In [11]:
# Convert the registered-voter change into a percentage growth rate for each ward.
# This makes voter-registration changes comparable across wards of different sizes.

ward_features["RegistrationGrowth"] = pd.NA

previous_registered = (
    ward_features.groupby("WardKey")["RegisteredVoters"]
    .shift(1)
)

ward_features.loc[safe_rows, "RegistrationGrowth"] = (
    ward_features.loc[safe_rows, "RegisteredVoters"]
    - previous_registered.loc[safe_rows]
) / previous_registered.loc[safe_rows] * 100

print("Registration growth populated:",
      ward_features["RegistrationGrowth"].notna().sum())

print("Registration growth blank:",
      ward_features["RegistrationGrowth"].isna().sum())

print("\nRegistration growth by election year:")
print(
    ward_features
    .groupby("ElectionYear")["RegistrationGrowth"]
    .apply(lambda x: x.notna().sum())
)

print("\n2011 registration growth populated:",
      ward_features.loc[
          ward_features["ElectionYear"] == 2011,
          "RegistrationGrowth"
      ].notna().sum())

Registration growth populated: 1434
Registration growth blank: 1165

Registration growth by election year:
ElectionYear
2011      0
2016    717
2021    717
Name: RegistrationGrowth, dtype: int64

2011 registration growth populated: 0


In [12]:
# One important point: RegistrationGrowth is expressed as a percentage, so later we can compare registration changes across wards of different sizes.

# Check that the historical turnout and registration features are populated together for safe ward comparisons.
# This confirms that no historical feature was calculated where the previous election was unreliable.

feature_columns = [
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth"
]

print("Feature population counts:")
print(
    ward_features[feature_columns]
    .notna()
    .sum()
)

print("\nPopulation by election year:")
print(
    ward_features
    .groupby("ElectionYear")[feature_columns]
    .apply(lambda x: x.notna().sum())
)

print("\nRows where all three historical features are populated:",
      ward_features[feature_columns].notna().all(axis=1).sum())

print("\nRows where historical features disagree:",
      (
          ward_features[feature_columns].notna().nunique(axis=1) > 1
      ).sum())

Feature population counts:
PreviousTurnout           1434
RegisteredVotersChange    1434
RegistrationGrowth        1434
dtype: int64

Population by election year:
              PreviousTurnout  RegisteredVotersChange  RegistrationGrowth
ElectionYear                                                             
2011                        0                       0                   0
2016                      717                     717                 717
2021                      717                     717                 717

Rows where all three historical features are populated: 1434

Rows where historical features disagree: 0


#=======================================================================================================================================================#
Now we create the naive baseline: each ward's latest historical turnout (2021) is carried forward as its predicted 2026 turnout.

This is not the ML model. It gives us a simple benchmark that the eventual model can be compared against.

Because the baseline is specifically a 2026 prediction, we will only use the 2021 historical turnout as its source. We'll also keep it blank for wards without a 2021 record.

In [13]:
# Carry each ward's latest historical turnout forward as a simple 2026 baseline prediction.
# This gives the final ML model a clear benchmark to improve on.

latest_turnout = (
    ward_features[
        ward_features["ElectionYear"] == 2021
    ]
    .set_index("WardKey")["TurnoutRate"]
)

ward_features["BaselinePredictedTurnout2026"] = (
    ward_features["WardKey"].map(latest_turnout)
)

print("2026 baseline populated:",
      ward_features["BaselinePredictedTurnout2026"].notna().sum())

print("2026 baseline blank:",
      ward_features["BaselinePredictedTurnout2026"].isna().sum())

print("\n2021 turnout records used:",
      len(latest_turnout))

print("\nBaseline range:")
print(
    ward_features["BaselinePredictedTurnout2026"]
    .dropna()
    .agg(["min", "max"])
)

2026 baseline populated: 2489
2026 baseline blank: 110

2021 turnout records used: 901

Baseline range:
min    16.877470
max    73.341864
Name: BaselinePredictedTurnout2026, dtype: float64


2026 baseline populated: 2489
2026 baseline blank: 110

2021 turnout records used: 901

Baseline range:
min    16.877470
max    73.341864
Name: BaselinePredictedTurnout2026, dtype: float64

In [14]:
# Confirm that every available 2026 baseline comes directly from the ward's 2021 turnout.
# This makes sure the baseline is a true historical benchmark and not a newly calculated prediction.

baseline_check = ward_features[
    ward_features["ElectionYear"] == 2021
].copy()

baseline_check["BaselineMatches2021"] = (
    baseline_check["BaselinePredictedTurnout2026"]
    == baseline_check["TurnoutRate"]
)

print("2021 baseline matches turnout:",
      baseline_check["BaselineMatches2021"].sum())

print("2021 baseline mismatches:",
      (~baseline_check["BaselineMatches2021"]).sum())

print("\n2021 rows with missing baseline:",
      baseline_check["BaselinePredictedTurnout2026"].isna().sum())

print("\nRows with no 2021 baseline source:",
      ward_features["BaselinePredictedTurnout2026"].isna().sum())

2021 baseline matches turnout: 901
2021 baseline mismatches: 0

2021 rows with missing baseline: 0

Rows with no 2021 baseline source: 110


0 mismatches. The baseline is correctly sourced from 2021 turnout.

One thing to keep clear in the notebook: the 110 blanks are expected because those ward keys do not have a 2021 historical record. We should not replace them with zero or another year's turnout.

In [15]:
# Calculate the KZN ward-level average turnout separately for each election year.
# This gives us the correct yearly benchmark for identifying below-average ward turnout.

provincial_average = (
    ward_features
    .groupby("ElectionYear")["TurnoutRate"]
    .transform("mean")
)

ward_features["ProvincialAverageTurnout"] = provincial_average

print("Provincial average turnout by election year:")
print(
    ward_features
    .groupby("ElectionYear")["ProvincialAverageTurnout"]
    .first()
)

print("\nMissing provincial averages:",
      ward_features["ProvincialAverageTurnout"].isna().sum())

Provincial average turnout by election year:
ElectionYear
2011    61.074435
2016    60.570788
2021    49.556774
Name: ProvincialAverageTurnout, dtype: float64

Missing provincial averages: 0


We now have the correct KZN ward-level benchmark for each election year:

Election year	KZN average turnout
2011	61.07%
2016	60.57%
2021	49.56%

No missing averages, so we can safely create the flag.

In [16]:
# Flag wards whose turnout was below the KZN average for that same election year.
# This captures whether a ward performed below its provincial turnout benchmark.

ward_features["BelowProvincialAverageTurnout"] = (
    ward_features["TurnoutRate"]
    < ward_features["ProvincialAverageTurnout"]
)

print("Below provincial average counts:")
print(
    ward_features["BelowProvincialAverageTurnout"]
    .value_counts()
)

print("\nBelow provincial average by election year:")
print(
    ward_features
    .groupby("ElectionYear")["BelowProvincialAverageTurnout"]
    .value_counts()
)

print("\nMissing flags:",
      ward_features["BelowProvincialAverageTurnout"].isna().sum())

Below provincial average counts:
BelowProvincialAverageTurnout
False    1369
True     1230
Name: count, dtype: int64

Below provincial average by election year:
ElectionYear  BelowProvincialAverageTurnout
2011          False                            434
              True                             394
2016          False                            437
              True                             433
2021          False                            498
              True                             403
Name: count, dtype: int64

Missing flags: 0


In [17]:
# Check whether both lower and upper poverty-line fields are available in the merged dataset.
# We document missing poverty-line fields instead of silently replacing them with another measure.

lbpl_columns = [
    column for column in ward_features.columns
    if "lbpl" in column.lower()
]

ubpl_columns = [
    column for column in ward_features.columns
    if "ubpl" in column.lower()
]

print("LBPL columns found:", lbpl_columns)
print("UBPL columns found:", ubpl_columns)

print("\nPoverty columns currently available:")
print([
    column for column in ward_features.columns
    if "poverty" in column.lower()
    or "food poverty" in column.lower()
])

print("\nLBPL available:", len(lbpl_columns) > 0)
print("UBPL available:", len(ubpl_columns) > 0)
print(
    "LBPL and UBPL pairing available:",
    len(lbpl_columns) > 0 and len(ubpl_columns) > 0
)

LBPL columns found: []
UBPL columns found: []

Poverty columns currently available:
['Food poverty headcount (FPL, %)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)']

LBPL available: False
UBPL available: False
LBPL and UBPL pairing available: False


In [18]:
# Review the new feature columns before the final validation and save.
# This confirms the feature-engineered dataset contains the intended historical and prediction features.

engineered_columns = [
    "PreviousElectionYear",
    "ExpectedPreviousYear",
    "ConsecutiveElection",
    "SafeHistoricalComparison",
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth",
    "BaselinePredictedTurnout2026",
    "ProvincialAverageTurnout",
    "BelowProvincialAverageTurnout"
]

print("Engineered columns:")
print(engineered_columns)

print("\nEngineered feature shape:",
      ward_features[engineered_columns].shape)

print("\nMissing values in engineered features:")
print(
    ward_features[engineered_columns]
    .isna()
    .sum()
)

Engineered columns:
['PreviousElectionYear', 'ExpectedPreviousYear', 'ConsecutiveElection', 'SafeHistoricalComparison', 'PreviousTurnout', 'RegisteredVotersChange', 'RegistrationGrowth', 'BaselinePredictedTurnout2026', 'ProvincialAverageTurnout', 'BelowProvincialAverageTurnout']

Engineered feature shape: (2599, 10)

Missing values in engineered features:
PreviousElectionYear             1011
ExpectedPreviousYear              828
ConsecutiveElection                 0
SafeHistoricalComparison            0
PreviousTurnout                  1165
RegisteredVotersChange           1165
RegistrationGrowth               1165
BaselinePredictedTurnout2026      110
ProvincialAverageTurnout            0
BelowProvincialAverageTurnout       0
dtype: int64


In [19]:
# Validate the final feature values before removing the temporary comparison controls.
# This checks row count, ward-year uniqueness, required features, and the original turnout data.

final_feature_columns = [
    "PreviousTurnout",
    "RegisteredVotersChange",
    "RegistrationGrowth",
    "BaselinePredictedTurnout2026",
    "ProvincialAverageTurnout",
    "BelowProvincialAverageTurnout"
]

print("Dataset shape:", ward_features.shape)

print("\nWard-year duplicates:",
      ward_features.duplicated(
          subset=["MunicipalityCode", "Ward", "ElectionYear"]
      ).sum())

print("\nRequired feature columns present:",
      all(
          column in ward_features.columns
          for column in final_feature_columns
      ))

print("\nOriginal turnout missing:",
      ward_features["TurnoutRate"].isna().sum())

print("\nFeature missing values:")
print(
    ward_features[final_feature_columns]
    .isna()
    .sum()
)

Dataset shape: (2599, 58)

Ward-year duplicates: 0

Required feature columns present: True

Original turnout missing: 0

Feature missing values:
PreviousTurnout                  1165
RegisteredVotersChange           1165
RegistrationGrowth               1165
BaselinePredictedTurnout2026      110
ProvincialAverageTurnout            0
BelowProvincialAverageTurnout       0
dtype: int64


In [20]:
# Remove only the temporary columns used to control historical feature calculations.
# Original master fields and the six final engineered features remain unchanged.

helper_columns = [
    "WardKey",
    "PreviousElectionYear",
    "ExpectedPreviousYear",
    "ConsecutiveElection",
    "SafeHistoricalComparison"
]

ward_features = ward_features.drop(
    columns=helper_columns
)

print("Removed helper columns:", helper_columns)

print("\nFinal dataset shape:", ward_features.shape)

print("\nFinal engineered features:")
print([
    column for column in ward_features.columns
    if column in final_feature_columns
])

Removed helper columns: ['WardKey', 'PreviousElectionYear', 'ExpectedPreviousYear', 'ConsecutiveElection', 'SafeHistoricalComparison']

Final dataset shape: (2599, 53)

Final engineered features:
['PreviousTurnout', 'RegisteredVotersChange', 'RegistrationGrowth', 'BaselinePredictedTurnout2026', 'ProvincialAverageTurnout', 'BelowProvincialAverageTurnout']


We now have the clean feature-engineered dataset at 2,599 rows × 53 columns.

The six intended engineered features are all present, and the temporary comparison controls are gone. Your original poverty/context fields remain untouched.

In [21]:
# Save the feature-engineered ward dataset as the next processed project layer.
# Reopening the file confirms the saved CSV keeps the same rows, columns, and ward-year structure.

ward_features.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

saved_features = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

print("Feature file exists:", output_path.exists())
print("Saved shape:", saved_features.shape)
print("Saved columns:", len(saved_features.columns))

print("\nWard-year duplicates after reopen:",
      saved_features.duplicated(
          subset=["MunicipalityCode", "Ward", "ElectionYear"]
      ).sum())

print("\nMissing required keys after reopen:",
      saved_features[
          ["Province", "Municipality", "Ward", "ElectionYear", "TurnoutRate"]
      ].isna().sum().sum())

print("\nFinal engineered features present:",
      all(
          column in saved_features.columns
          for column in final_feature_columns
      ))

print("\nFEATURE ENGINEERING SAVE VALIDATION:",
      "PASSED"
      if (
          output_path.exists()
          and saved_features.shape == ward_features.shape
          and len(saved_features.columns) == len(ward_features.columns)
          and saved_features.duplicated(
              subset=["MunicipalityCode", "Ward", "ElectionYear"]
          ).sum() == 0
          and saved_features[
              ["Province", "Municipality", "Ward", "ElectionYear", "TurnoutRate"]
          ].isna().sum().sum() == 0
          and all(
              column in saved_features.columns
              for column in final_feature_columns
          )
      )
      else "FAILED"
)

Feature file exists: True
Saved shape: (2599, 53)
Saved columns: 53

Ward-year duplicates after reopen: 0

Missing required keys after reopen: 0

Final engineered features present: True

FEATURE ENGINEERING SAVE VALIDATION: PASSED
